# ReAct Agent from Scratch

Companion notebook for the [Agent Architectures](https://codemia.ai/courses/applied_ai_systems/agent_architectures) lesson on Codemia.

**What you will build**: a ReAct agent in about 120 lines of standard Python: a tool registry, a thought-action-observation loop, and termination logic. No LangChain, no LlamaIndex, no agent framework of any kind.

**Why from scratch**: every agent framework is a loop around a model call plus a dispatch table. Writing it yourself makes the three failure modes visible: the model that never calls a tool, the model that never stops calling them, and the tool whose output the model cannot use.

**References**:
- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (2022)
- OpenAI, [Function calling guide](https://platform.openai.com/docs/guides/function-calling)

**Time**: ~45 minutes  **GPU required**: no

## Setup

The notebook runs in two modes.

With an `OPENAI_API_KEY` set you get the real thing: a function-calling model
deciding which tool to reach for. Without one it falls back to a scripted stand-in
so the loop, the trace and the termination logic are all still observable. The
stand-in is not a model and is not pretending to be one, it is a fixed policy that
lets you run every cell below on a machine with no credentials.

In [ ]:
%pip install -q openai

In [ ]:
import json
import os
import re
import math
import textwrap

USE_REAL_LLM = bool(os.environ.get('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('mode:', 'live model' if USE_REAL_LLM else 'scripted stand-in')

## Concept recap

ReAct interleaves two things the model was already able to do separately: write
down a thought, and emit a structured action. One step of the loop is

    thought  ->  action (a tool call)  ->  observation (the tool result)

and the observation is appended to the transcript so the next thought can use it.
That is the whole idea. What makes it work in practice is not the prompt, it is the
loop around it: something has to decide when the agent is done, and something has
to stop it when it is not converging.

Three parts, and we build them in this order:

1. **Tools**: plain Python functions plus a JSON schema the model can read.
2. **The model call**: messages in, either a tool call or a final answer out.
3. **The loop**: dispatch, append the observation, repeat, and stop.

## Tools

A tool is a function and a schema. Keep the function boring and the schema honest:
the description is the only documentation the model gets, so it is part of the
implementation, not a comment.

Our agent gets two tools, a document search over a tiny fixed corpus and a
calculator, chosen so that one question needs both.

In [ ]:
CORPUS = {
    'orca-3b': 'Orca-3B is a 3.1 billion parameter model released in March 2024. '
               'It scores 61.4 on MMLU and costs 0.18 USD per million input tokens.',
    'orca-7b': 'Orca-7B has 7.2 billion parameters, released June 2024. '
               'It scores 68.9 on MMLU and costs 0.55 USD per million input tokens.',
    'serving': 'Orca models are served at 1,400 tokens per second per A100 for the '
               '3B and 520 tokens per second per A100 for the 7B.',
    'licence': 'Both Orca models ship under Apache 2.0 and permit commercial use.',
}


def search_docs(query: str) -> str:
    # Deliberately dumb retrieval: lowercase token overlap. A real agent would call
    # a vector index here, but the agent loop cannot tell the difference, and that
    # is the point. Swapping this for real retrieval changes no other line.
    q = set(re.findall(r'[a-z0-9.]+', query.lower()))
    scored = [(len(q & set(re.findall(r'[a-z0-9.]+', (k + ' ' + v).lower()))), k, v)
              for k, v in CORPUS.items()]
    scored.sort(reverse=True)
    hits = [f'[{k}] {v}' for n, k, v in scored[:2] if n > 0]
    return '\n'.join(hits) if hits else 'No matching documents.'

In [ ]:
def calculator(expression: str) -> str:
    # eval on model output is a real risk, so the grammar is restricted to arithmetic
    # before anything is evaluated. An agent that can run arbitrary Python is a
    # different notebook with a sandbox in it.
    if not re.fullmatch(r'[0-9pi.e+\-*/() ]+', expression):
        return 'Error: only arithmetic is allowed.'
    try:
        value = eval(expression, {'__builtins__': {}}, {'pi': math.pi, 'e': math.e})
    except Exception as exc:
        return f'Error: {exc}'
    return f'{value:.6g}'

In [ ]:
TOOLS = {'search_docs': search_docs, 'calculator': calculator}

SCHEMAS = [
    {'type': 'function', 'function': {
        'name': 'search_docs',
        'description': 'Search the internal Orca model documentation. Returns up to '
                       'two matching passages. Use this for any factual claim about '
                       'model size, cost, benchmark scores, throughput or licence.',
        'parameters': {'type': 'object', 'properties': {
            'query': {'type': 'string', 'description': 'Keywords to search for.'}},
            'required': ['query']}}},
    {'type': 'function', 'function': {
        'name': 'calculator',
        'description': 'Evaluate an arithmetic expression. Use this instead of doing '
                       'arithmetic yourself, including simple arithmetic.',
        'parameters': {'type': 'object', 'properties': {
            'expression': {'type': 'string',
                           'description': 'An arithmetic expression, e.g. 0.55 / 0.18'}},
            'required': ['expression']}}},
]
print(json.dumps(SCHEMAS[1], indent=1)[:260])

## The model call

One function, one job: messages in, a decision out. We normalise both backends to
the same return shape, `(tool_calls, text)`, so the loop below never learns which
one it is talking to.

Note what the system prompt does and does not do. It tells the model to think before
acting and to ground every number in a tool result. It does not describe the tools,
because the schemas already do that, and a prose restatement of a schema is one more
thing to keep in sync.

In [ ]:
SYSTEM = (
    'You are a careful research assistant. Work in small steps. Before each tool '
    'call, state in one sentence what you are trying to learn. Never state a number '
    'that did not come from a tool result. When you have enough to answer, answer '
    'directly and do not call another tool.'
)


def call_live(messages):
    from openai import OpenAI
    rsp = OpenAI().chat.completions.create(
        model=MODEL, messages=messages, tools=SCHEMAS, temperature=0)
    msg = rsp.choices[0].message
    calls = [(c.id, c.function.name, json.loads(c.function.arguments))
             for c in (msg.tool_calls or [])]
    return calls, (msg.content or '')

In [ ]:
def answer_from_observations(question, observed):
    # Answer only out of a retrieved sentence that actually mentions the attribute
    # that was asked about. This is the line that decides whether the stand-in
    # declines or invents, and it is the same decision the system prompt is asking
    # a real model to make.
    key = next((k for k in ('licence', 'mmlu', 'tokens per second', 'parameters',
                            'context window', 'commercial') if k in question), None)
    sentence = next((s for s in re.split(r'(?<=\.)\s+', observed)
                     if key and key in s.lower()), None)
    if sentence:
        return f'From the documentation: {sentence.strip()}'
    return 'The documentation I retrieved does not state that, so I will not guess.'

In [ ]:
def call_scripted(messages):
    # A fixed policy, not a model. It reads the transcript and picks the next obvious
    # move, quoting documented figures by name rather than by position.
    used = [m.get('name') for m in messages if m['role'] == 'tool']
    question = messages[1]['content'].lower()
    obs = [m['content'] for m in messages if m['role'] == 'tool']
    observed = '\n'.join(obs)

    if 'search_docs' not in used:
        return ([('c1', 'search_docs', {'query': messages[1]['content']})],
                'I need the documented figures before I say anything numeric.')

    costs = dict(re.findall(r'\[(orca-\d+b)\][^\n]*?costs (\d+\.\d+) USD', observed))
    if 'more expensive' in question:
        if 'calculator' not in used:
            if len(costs) < 2:
                return [], 'The retrieved passages do not contain both per-token costs.'
            return ([('c2', 'calculator',
                      {'expression': f'{costs["orca-7b"]} / {costs["orca-3b"]}'})],
                    'Now I divide the two documented per-token costs.')
        return [], (f'Orca-7B costs {costs["orca-7b"]} USD per million input tokens '
                    f'against {costs["orca-3b"]} for Orca-3B, a factor of {obs[-1]}.')

    return [], answer_from_observations(question, observed)


call_llm = call_live if USE_REAL_LLM else call_scripted

## The loop

Everything above was setup. This is the agent.

Four things happen per iteration, and three of them are bookkeeping:

- ask the model what to do next,
- if it asked for tools, run them and append each result as a `tool` message,
- if it answered instead, stop,
- and if neither of those has happened by `max_steps`, stop anyway.

That last clause is the one people leave out. A loop with no step budget is not an
agent, it is an unbounded bill.

In [ ]:
def run_agent(question, max_steps=6, verbose=True):
    messages = [{'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': question}]
    trace = []
    for step in range(1, max_steps + 1):
        calls, text = call_llm(messages)
        if text:
            trace.append(('thought', text))
        if not calls:
            # The model answered instead of acting. This is the normal exit.
            trace.append(('answer', text))
            return {'answer': text, 'steps': step, 'stopped': 'answered', 'trace': trace}
        messages.append({'role': 'assistant', 'content': text or None,
                         'tool_calls': [{'id': i, 'type': 'function',
                                         'function': {'name': n,
                                                      'arguments': json.dumps(a)}}
                                        for i, n, a in calls]})
        for cid, name, args in calls:
            result = TOOLS[name](**args) if name in TOOLS else f'Error: no tool {name}'
            trace.append(('action', f'{name}({json.dumps(args)})'))
            trace.append(('observation', result))
            messages.append({'role': 'tool', 'tool_call_id': cid, 'name': name,
                             'content': result})
    # Budget exhausted. Returning the partial transcript beats raising, because the
    # caller usually wants to show the user what the agent did manage to find.
    return {'answer': None, 'steps': max_steps, 'stopped': 'step budget', 'trace': trace}

In [ ]:
def show(result):
    label = {'thought': 'THOUGHT', 'action': 'ACTION', 'observation': 'OBSERVE',
             'answer': 'ANSWER'}
    for kind, body in result['trace']:
        if kind == 'answer':
            continue
        for i, line in enumerate(textwrap.wrap(body, 86) or ['']):
            print(f'{label[kind] if i == 0 else "":9s} {line}')
    print('-' * 96)
    print(f'stopped: {result["stopped"]}  after {result["steps"]} step(s)')
    print(f'answer : {result["answer"]}')

## Experiment

Three questions, picked to land on three different paths through the loop.

The first needs one search. The second needs a search and then arithmetic on what
the search returned, which is the case that justifies the loop existing at all: the
second tool call cannot be planned until the first observation arrives. The third is
not in the corpus, which is where you find out whether your agent says so or invents
an answer.

In [ ]:
r1 = run_agent('What is the licence on the Orca models?')
show(r1)

In [ ]:
r2 = run_agent('How many times more expensive per input token is Orca-7B than Orca-3B?')
show(r2)

In [ ]:
r3 = run_agent('What is the context window of Orca-7B?')
show(r3)

## Variations

Three ablations. Each one removes a single part of the loop, and each one breaks it
in a different way, which is a more useful thing to see than a working trace.

In [ ]:
# 1. Starve the loop of observations. The tool still runs, but its result never
# reaches the transcript, so the next thought has nothing new to work with.
# The assistant tool_calls message is still appended: a tool message with no
# tool_calls message before it is a 400 from the API, not an ablation.
def run_without_observations(question, max_steps=4):
    messages = [{'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': question}]
    for step in range(max_steps):
        calls, text = call_llm(messages)
        if not calls:
            return f'answered at step {step + 1}: {text[:90]}'
        messages.append({'role': 'assistant', 'content': text or None,
                         'tool_calls': [{'id': i, 'type': 'function',
                                         'function': {'name': n,
                                                      'arguments': json.dumps(a)}}
                                        for i, n, a in calls]})
        for cid, name, args in calls:
            TOOLS[name](**args)   # result computed, then dropped on the floor
            messages.append({'role': 'tool', 'tool_call_id': cid, 'name': name,
                             'content': '(observation withheld)'})
    return f'never answered, hit the {max_steps} step budget'


print(run_without_observations('How many times more expensive per input token is '
                               'Orca-7B than Orca-3B?'))

In [ ]:
# 2. Shrink the step budget below what the task needs. The two-tool question needs
# three model calls; give it two and watch where it lands.
for budget in (1, 2, 3, 6):
    r = run_agent('How many times more expensive per input token is Orca-7B '
                  'than Orca-3B?', max_steps=budget)
    print(f'max_steps={budget}  stopped={r["stopped"]:12s}  answer={str(r["answer"])[:54]}')

In [ ]:
# 3. Count tool calls per question. This is the number that shows up on the bill,
# and the number a step budget is actually bounding.
for q in ['What is the licence on the Orca models?',
          'How many times more expensive per input token is Orca-7B than Orca-3B?',
          'What is the context window of Orca-7B?']:
    r = run_agent(q)
    n = sum(1 for kind, _ in r['trace'] if kind == 'action')
    print(f'{n} tool call(s), stopped by {r["stopped"]:12s} | {q[:58]}')

Things worth noticing in the traces above:

1. **The second tool call is not plannable.** The division in question two depends on
   two numbers that only exist after the search returns. No amount of up-front
   planning produces that call, which is the argument for a loop over a pipeline.
2. **Withholding observations does not raise an error.** The agent keeps calling
   tools and keeps getting nowhere, and the only signal that something is wrong is
   that it hits the step budget. Silent non-convergence is the normal shape of a
   broken agent.
3. **A too-small budget fails quietly too.** It returns `stopped: step budget` with a
   `None` answer. If your caller treats a missing answer as an empty string, that
   turns into a blank response in production and nothing in the logs.
4. **The unanswerable question is the one that tests the system prompt.** Grounding
   every number in a tool result is the instruction that makes the model say it
   cannot find something, instead of producing a plausible context window.

## What to try next

- Add a third tool with an overlapping description, say `lookup_pricing`, and see how
  often the model picks the wrong one. Overlapping tool descriptions are the most
  common cause of bad dispatch, and they get worse as the registry grows.
- Make a tool fail. Have `search_docs` raise on every third call and decide whether
  the observation should carry the traceback, a terse message, or a retry hint. The
  wording of a failed observation changes the recovery behaviour more than you expect.
- Add a cost ceiling next to the step ceiling: accumulate `usage.total_tokens` from
  each response and stop on either bound, whichever binds first.
- Replace the step budget with a convergence check. Stop when two consecutive steps
  issue the same tool call with the same arguments, which is the signature of a stuck
  loop and is cheaper to detect than to wait out.
- Swap `search_docs` for the HNSW index from the
  [Vector Search Algorithms](https://codemia.ai/courses/applied_ai_systems/vector_search_algorithms)
  notebook. Nothing in the loop changes, which is the test of whether your tool
  boundary is drawn in the right place.